Hver fil må dekodes i ISO-8859-1/LATIN1 eller UTF-8, kanskje kan det sjekkes ved at hvis dekodingen returnerer første tegn som er riktig returnerer funksjonen True, og resten av filen leses med gitt dekoding. Hvis ikke så testes den andre dekodingen

Filtrer ut støy ved å si at alle linjer som ikke begynner med '[' er støy.
Videre kan hver linje legges til i ei liste hvor elementene per linje er adskilt med 'mellomrom'


Basert på oppgaveteksten så virker det som priser i EUR og USD er korrekte, med at kjøp gjennomført i NOK må konverteres fra EUR til NOK da verdien egentlig står i EUR.

TASK 1

In [11]:

from pathlib import Path
import json
import numpy as np
import math as m
from typing import Any

In [12]:

def find_encoding(file: Path):
    """Checks if encoding of file is UTF-8, if not, sets encoding to Latin 1"""
    file = Path(file)
    try:
        with file.open(encoding= 'utf-8') as f:
            f.read()
        Etype = 'utf-8'
    
    except UnicodeDecodeError:
        Etype = 'iso-8859-1'

    return Etype

        
# I had to alter one of the elements in the 'Cgenerate_data_d2.py' because the vreated files only contained ASCII chars,
#so it wasnt possible to separate utf-8 from latin 1 without æ,ø,å. To solve, 'muffin' was changed to 'møffin'

a =find_encoding('C:/Users/danie/Documents/NMBU 4.år/Inf 201/Deliverables/W43_44/INF201_Deliverable_2/2026/library_cafe/POS_01/register_sales.log')
b=find_encoding('2026/library_cafe/POS_02/register_sales.log')
c=find_encoding('C:/Users/danie/Documents/NMBU 4.år/Inf 201/Deliverables/W43_44/INF201_Deliverable_2/2026/library_cafe/POS_03/register_sales.log')
d=find_encoding('C:/Users/danie/Documents/NMBU 4.år/Inf 201/Deliverables/W43_44/INF201_Deliverable_2/2026/science_building/POS_02/register_sales.log')
print(a, b, c, d)



FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\danie\\Documents\\NMBU 4.år\\Inf 201\\Deliverables\\W43_44\\INF201_Deliverable_2\\2026\\library_cafe\\POS_01\\register_sales.log'

Write a log parser function that uses regular expressions. Log files contain valid sales
lines mixed with system noise.
Valid line format:
[2026-10-15 08:30:00] [POS_01] item=Latte price=4.50 EUR (SUCCESS)
Noise lines to ignore:
--- PRINTER PAPER LOW WARNING ---
=== NETWORK SYNC OK ===
Construct a regular expression ( re module) to capture the 6 key fields from
valid lines:
timestamp (string, format: YYYY-MM-DD HH:MM:SS )
register_id (string, e.g., POS_01 )
item (string, e.g., Latte )
price (float, e.g., 4.50 )
currency (string, e.g., EUR , NOK , USD )
status (string: SUCCESS , REFUND , or CANCELLED )
Return a list of Python dictionaries, where each dictionary represents one valid
transaction record.

In [ ]:


def log_parser(file: Path)->list[dict]:
    """ Takes registerlogs as input, returns a list where each element is a dictionary of successfull transactions. """
    valid_lines = []
    transactions = [] 
    #opening file with correct input
    with open(file, encoding=find_encoding(file)) as f:
        line = f.readlines() #readlines create a list,line where each line in f becomes a object in that list.

    #step 1: removing all noise lines.
    for i in line:
        if i[0:2] =='[2':
            valid_lines.append(i) #if the object of line begins with [2, the first to chars of the time stamp of valid lines
    
    #step 2: Creating dictionaries for all valid lines, append them to a list of dictionaries    
    for valid_line in valid_lines:
        elements = valid_line.split()
        
        transaction = {
            'timestamp': elements[0].strip('[') +' '+ elements[1].strip(']'),
            'register' : elements[2].strip('[]'), 
            'item' : elements[3].split('=')[1],
            'price' : float(elements[4].split('=')[1]),
            'currency' : elements[5].strip(),
            'status' : elements[6].strip('()')
        }
        transactions.append(transaction)
    
    return transactions #return only the three first for visualiaztion



def valid_transactions(liste: list[dict]) -> list[dict]:
    """Filtering out only the valid transactions from log_parser"""
    return [transaction for transaction in liste if transaction['status'] == 'SUCCESS']



valid_transactions(log_parser('2026/library_cafe/POS_01/register_sales.log'))



Recap task 1: 
We see that using only Path and regular expressions we've created a set of functions that recieve raw log data from cashregisters, filters out noise lines and then sorts out only the successfull transactions. These transactions are then (as individual dictionaries) added to a list. The filtering for sorting out the lines that begin with '[2' and '[D', is maybe a bit primitive, but it will work until th timstamp hits th year 3000.

TASK 2

1. Write a recursive directory search function that finds all log files in a nested directory
and returns a list of pathlib.Path objects:
Convert the input path to a pathlib.Path object.
Iterate through items using Path.iterdir() .
If an item is a directory, recursively search inside it.
If an item is a file with a matching extension ( .log ), add its Path object to a
list.
Constraint: Do NOT use Path.glob() , Path.rglob() , or os.walk() . You
must implement explicit recursion.
2. Metadata about the different stores can be found in the
data/store_config.json file. This file contains deeply nested dictionaries and
lists (see example below). Write a recursive function that:
Traverses any combination of nested dictionaries and lists found in this file.
Returns a set (type hint: set[str] ) of all register id strings where "active":
true 

In [ ]:



def dir_search(dirpath: str) -> list[str]:
    """Search through a directory. Add all paths of '.log' - files to a list. """
    log_files = []
    dirpath = Path(dirpath)
    # 1. searching through input directory
    for file in dirpath.iterdir():
        if file.is_file() and file.suffix == '.log':
            log_files.append(file)
      
        elif file.is_dir():
            #collects the logfiles if they are in subdirectories
            #if log_files list is local to the func we must do it this way in order to not clear out list in every recursive step
            log_files.extend(dir_search(file))
        
    return log_files


def find_active(data: Any) -> set[str]:
    """Returns at set of strings containing register name and whether it is active or not"""
    active_ids: set[str] = set()

    if isinstance(data, dict):
        # Check if current dir is active register
        if data.get('active') is True and 'id' in data:
            active_ids.add(str(data['id']))
        for value in data.values():
            active_ids.update(find_active(value))

    elif isinstance(data, list):
        #Check each list
        for item in data:
            active_ids.update(find_active(item))

    return active_ids




with open('store_config.json', 'r') as f:
    config_data = json.load(f)

find_active(config_data)

print(find_active(config_data),dir_search('2026')  )

dir_search('2026')




TASK 3

 Write a function for removing price outliers that:
Converts input price data to a 1D NumPy float array.
Detects hardware glitch outliers ( ) where a value deviates from the mean
by more than 3 standard deviations:
Replaces detected outlier values with the median of the valid prices.
Returns the cleaned 1D array.
2. Write a function compute_refund_ratio(sales_count: np.ndarray,
refund_count: np.ndarray) -> np.ndarray that:
Computes the element-wise refund ratio using NumPy array operations:
The addition of prevents zero-division errors.
Task 4: Data analysis using Panda

In [ ]:
"""TASK3.1"""


def mean(array: np.array) -> float:
    """Returns mean value of numpy array"""
    return float(sum(array)/len(array))


def stdev(array: np.array) -> float:
    """Returns standard deviation of array"""
    x_mean = mean(array)
    variance = sum((x-x_mean)**2 for x in array) /len(array)
    return float(m.sqrt(variance))


def valid_prices(dir):
    """Takes a directory as input, returns prices from all transactions"""
    prices=[]
    filepaths = dir_search(dir)
    print(filepaths)

    for file_path in filepaths:
        transactions = log_parser(file_path)
        for trans in transactions:
            if 'price' in trans:
                prices.append(trans['price'])
    price_array = np.array(prices, dtype=float)
    if price_array.size == 0:
        return price_array

    x_mean = mean(price_array)
    x_stdev = stdev(price_array)
    #create boolean values for if price is an outlier or not.
    valid_price_filter = np.abs(price_array - x_mean) <= 3*x_stdev

    clean_prices = price_array[valid_price_filter]
    return clean_prices


    
    
a = valid_prices('2026')
print(a)






In [ ]:
"""TASK 3.2"""

def get_items(filepaths):
        """Creates a list of the items in registerlogs"""
        items_set = set()

        for file_path in filepaths:
            transactions = log_parser(file_path)
            for transaction in transactions:
                if 'item' in transaction:
                    items_set.add(transaction['item'])
        return sorted(list(items_set))

def sale_refund_arrays(filepaths) -> np.ndarray:
    """Counts the sales and refunds, returns to arrays"""
    items = get_items(filepaths)
    print(f'Found items_ {items}')

    sales_dict = {item: 0 for item in items}
    refund_dict = {item: 0 for item in items}

    for file_path in filepaths:
         transactions = log_parser(file_path)
         for transaction in transactions:
              item = transaction.get('item')
              status = transaction.get('status')

              if item in sales_dict:
                    if status == 'SUCCESS':
                        sales_dict[item] += 1
                    elif status == 'REFUND':
                        refund_dict[item] += 1
    sales_counter = np.array([sales_dict[item] for item in items], dtype = float)
    refund_counter = np.array([refund_dict[item] for item in items], dtype=float)
    

    return sales_counter, refund_counter
    



def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    """Computes refund ratio based on count8 of refund and successes"""

    #add a base value in the denominator to avoid divide by zero errors
    base = 1e-6

    total_sales = sum(sales_count)
    total_refunds = sum(refund_count)
    total_ratio = total_refunds/ (total_sales + base)

    return f""" Total Refund/Sales ratio = {total_ratio}
    Refund ratio per item in found order = {refund_count/(sales_count + base)})"""


ratios = compute_refund_ratio(*sale_refund_arrays(dir_search('2026')))
print(ratios)



In [ ]:
import pandas as pd  

# task 4
def create_transaction_dataframe(transactions: list[dict]) -> pd.DataFrame:
    df = pd.DataFrame(transactions)
    if not df.empty and 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df.set_index('timestamp', inplace=True)
        df.sort_index(inplace=True)
    return df

def resample_hourly_revenue(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    return df.groupby('item').resample('1h')['price'].sum().unstack(level=0, fill_value=0)

def status_revenue_pivot(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    return df.pivot_table(index='item', columns='status', values='price', aggfunc='sum', fill_value=0)

def calculate_item_refund_ratios(df: pd.DataFrame) -> pd.Series:
    if df.empty:
        return pd.Series()
    
    success_counts = df[df['status'] == 'SUCCESS'].groupby('item')['price'].count()
    refund_counts = df[df['status'] == 'REFUND'].groupby('item')['price'].count()
    summary_df = pd.DataFrame({'sales': success_counts, 'refunds': refund_counts}).fillna(0) # Riktig innrykk
    
    base = 1e-6
    ratios = summary_df['refunds'] / (summary_df['sales'] + base)
    return ratios


# task 5
import json

def run_coffee_shop_pipeline(root_dir: str, config_path: str):
    log_files = dir_search(root_dir)
    
    with open(config_path, 'r', encoding='utf-8') as f:
        config_data = json.load(f)
    active_registers = find_active(config_data)
    
    all_transactions = []
    for file_path in log_files:
        parsed_items = log_parser(file_path)
        for trans in parsed_items:
            if trans.get('register') in active_registers:
                all_transactions.append(trans)
                
    df = create_transaction_dataframe(all_transactions)
    
    if not df.empty:
        
        price_array = df['price'].to_numpy(dtype=float)
        x_mean = np.mean(price_array)
        x_stdev = np.std(price_array)
        median_val = np.median(price_array[np.abs(price_array - x_mean) <= 3 * x_stdev])
        
        outlier_mask = np.abs(price_array - x_mean) > 3 * x_stdev
        price_array[outlier_mask] = median_val
        df['price'] = price_array
        
        df.loc[df['currency'] == 'NOK', 'price'] *= 10.0
      
        hourly_rev = resample_hourly_revenue(df)
        pivot_table = status_revenue_pivot(df)
        refund_ratios = calculate_item_refund_ratios(df)
        
        df.reset_index().to_csv('cleaned_transactions.csv', index=False)
        hourly_rev.to_json('hourly_revenue.json', orient='index')
        pivot_table.to_csv('status_pivot_table.csv')
        refund_ratios.to_csv('item_refund_ratios.csv', header=['refund_ratio'])
        
        print("Arbeidsflyten er fullført! Alllefiler er generert.")

run_coffee_shop_pipeline('2026', 'store_config.json')

ModuleNotFoundError: No module named 'pandas'

In [ ]:
import ensurepip
ensurepip.bootstrap()



In [ ]:
!python -m pip install pandas numpy


   ---------------------------------------- 0.0/11.0 MB ? eta -:--:--
   ----- ---------------------------------- 1.6/11.0 MB 34.2 MB/s eta 0:00:01
   ----------------- ---------------------- 4.7/11.0 MB 49.7 MB/s eta 0:00:01
   ----------------------- ---------------- 6.3/11.0 MB 45.2 MB/s eta 0:00:01
   ---------------------------------- ----- 9.5/11.0 MB 50.5 MB/s eta 0:00:01
   ---------------------------------------- 11.0/11.0 MB 54.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   -------- ------------------------------- 2.8/12.6 MB 59.9 MB/s eta 0:00:01
   ------------------ --------------------- 5.8/12.6 MB 62.0 MB/s eta 0:00:01
   ---------------------------- ----------- 8.8/12.6 MB 62.7 MB/s eta 0:00:01
   ------------------------------------ --- 11.6/12.6 MB 65.6 MB/s eta 0:00:01
   ---------------------------------------- 12.6/12.6 MB 54.4 MB/s eta 0:00:00


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip
